### Import Required Libraries

In [1]:
import pandas as pd

### Import the DataSet

In [2]:
data = pd.read_csv('Data/data.csv')

### Extract Information of DataSet

In [3]:
data.head()

,Flags,Utterance,Category,Intent
0,BM,I Have Problems With Canceling An Order,Order,Cancel_Order
1,BIM,How Can I Find Information About Canceling Ord...,Order,Cancel_Order
2,B,I Need Help With Canceling The Last Order,Order,Cancel_Order
3,BIP,Could You Help Me Cancelling The Last Order I ...,Order,Cancel_Order
4,B,Problem With Cancelling An Order I Made,Order,Cancel_Order


### Checking the Null Values

In [4]:
data.isna().sum()

Flags        0
Utterance    0
Category     0
Intent       0
dtype: int64

### Separate DataSet Into Dependent and Independent Variable

In [5]:
X = data['Utterance']
y = data.drop('Utterance', axis = 1)

### Split DataSet into Training and Testing Variable

In [6]:
from sklearn.model_selection import train_test_split
def spliter(X, y):
    return train_test_split(X, y, random_state = 20, test_size = 0.2)

### Apply the Pipline Algorithm

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

#### For Intent

In [8]:
from sklearn.linear_model import LogisticRegression
model_Intent = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words = 'english', ngram_range=(1,2), max_df=0.9, min_df=2, sublinear_tf=True)),
    ('LogRegg', LogisticRegression())
     ])

#### For Category

In [9]:
from sklearn.svm import LinearSVC
model_Category = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words = 'english', ngram_range=(1,2), max_df=0.9, min_df=2, sublinear_tf=True)),
    ('LinSVC', LinearSVC())
     ])

#### For Flags

In [10]:
from sklearn.multiclass import OneVsRestClassifier
model_Flags = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words = 'english', ngram_range=(1,2), max_df=0.9, min_df=2, sublinear_tf=True)),
    ('LinSVC', OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=5000)))
     ])

### Training the Model

In [11]:
def training(model, X_train, y_train):
    model.fit(X_train, y_train)

### Calculating the Accuracy score of this Model

In [12]:
from sklearn.metrics import f1_score
def acc(model, X_test, y_test):
    y_pred = model.predict(X_test)
    accuracy = f1_score(y_pred, y_test, average='weighted')
    return round(accuracy*100,2)

### Model Dictionary

In [13]:
model = {
    'Flags': model_Flags,
    'Category': model_Category,
    'Intent': model_Intent
}

### Training and Checking the Accuracy of  Model

In [14]:
for i in y.columns:
    X_train, X_test, y_train, y_test = spliter(X, y[i])
    training(model[i], X_train, y_train)
    print(f"Accuracy Score of {i}'s Model is {acc(model[i], X_test, y_test)}%")

Accuracy Score of Flags's Model is 47.97%
Accuracy Score of Category's Model is 99.72%
Accuracy Score of Intent's Model is 97.74%


### Save the Model 

In [15]:
import pickle
def saved(model, name):
    pickle.dump(model, open(f'Model/{name}.pkl', 'wb'))
    print(f'{name} Model Saved Sucessfull')

In [17]:
for i in y.columns:
    saved(model[i], i)

Flags Model Saved Sucessfull
Category Model Saved Sucessfull
Intent Model Saved Sucessfull
